# 02 — Topological Sort and Disjoint Set Union (DSU)

## Why This Matters for DSA
In the previous notebook, we covered the basic representations and traversals of graphs. Now, we look at two major structural techniques: **Topological Sort** and **Disjoint Set Union (DSU)**. 

Topological Sorting answers: *"In what order must we perform dependent tasks?"* (e.g., compilation of files, scheduling tasks, resolving dependencies). Disjoint Set Union answers: *"Are these two elements in the same connected component, and how can we merge components efficiently in near $O(1)$ time?"* DSU is the primary engine behind Kruskal's Minimum Spanning Tree algorithm, dynamic connectivity, and transitive equivalence problems.

## Prerequisites
- `01_Representations_DFS_BFS` (Phase 04) — Adjacency lists, basic DFS recursion, and cycle detection concepts.

## Learning Objectives
By the end of this notebook, you will be able to:
- Implement Topological Sort using Kahn's Algorithm (BFS indegree-based).
- Implement Topological Sort using DFS with 3-state cycle detection.
- Implement a Disjoint Set Union (DSU) class with Path Compression and Union by Size/Rank.
- State the amortized $O(\alpha(N))$ complexity of DSU operations and explain the physical meaning of the Inverse Ackermann function.
- Detect cycles in undirected graphs dynamically using DSU.
- Benchmark DSU configurations to empirically observe the performance delta of path compression.

## Checklist Before Moving On
- [ ] I can write Kahn's algorithm and explain how it detects cycles in directed graphs.
- [ ] I can write a DFS-based topological sort using 3-state coloring to prevent invalid output on cyclic graphs.
- [ ] I can construct a DSU class from memory with both path compression and union by size.
- [ ] I can explain why the naive union-find can degenerate to $O(N)$ lookup times.
- [ ] I can use DSU to identify undirected cycles on the fly as edges are added.

## Table of Contents
1. Topological Sort — Kahn's Algorithm (BFS)
2. Topological Sort — DFS-Based Approach
3. Disjoint Set Union (DSU) — Core Operations
4. DSU Applications — Undirected Cycle Detection
5. Benchmarking DSU — Path Compression Impact
6. Checkpoint, Mini Project, and Practice Problems
7. LeetCode Practice Problems

## 1. Topological Sort — Kahn's Algorithm (BFS)

### The Why
Topological sort is a linear ordering of vertices in a Directed Acyclic Graph (DAG) such that for every directed edge $u \rightarrow v$, vertex $u$ comes before $v$ in the ordering. If a graph contains a cycle (e.g. $A \rightarrow B \rightarrow C \rightarrow A$), no topological order is possible because of circular dependencies.

Kahn's Algorithm is a BFS-based approach that simulates resolving dependencies step-by-step. It is highly intuitive: you can only process a task if all its prerequisites have already been completed.

### Core Mechanism
1. **Indegree Calculation:** Compute the number of incoming edges (prerequisites) for each node.
2. **Queue Initialization:** Push all nodes with an indegree of `0` (tasks with zero prerequisites) into a queue.
3. **Traversal:** While the queue is not empty:
   - Pop a node $u$ and append it to our sorted results.
   - For each neighbor $v$ of $u$, decrement its indegree by 1 (simulating the completion of prerequisite $u$).
   - If $v$'s indegree reaches `0`, push $v$ into the queue.
4. **Cycle Detection:** If the final sorted list contains fewer than $V$ elements, it means some nodes were never queueable because they were trapped in circular dependencies. Hence, a cycle exists.

### Common Pitfalls
- **Running on Undirected Graphs:** Topological sorting is *only* defined for Directed Acyclic Graphs. Running Kahn's algorithm on undirected graphs will immediately fail, as all connected nodes will have indegrees $\ge 1$ from the start.

In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <queue>
#include <algorithm>

using namespace std;

class KahnGraph {
    int V;
    vector<vector<int>> adj;

public:
    KahnGraph(int vertices) : V(vertices) {
        adj.resize(V);
    }

    void addEdge(int u, int v) {
        adj[u].push_back(v); // directed edge: u -> v
    }

    // Returns empty vector if cycle is detected
    vector<int> topologicalSort() {
        vector<int> inDegree(V, 0);
        for (int u = 0; u < V; ++u) {
            for (int v : adj[u]) {
                inDegree[v]++;
            }
        }

        queue<int> q;
        for (int i = 0; i < V; ++i) {
            if (inDegree[i] == 0) {
                q.push(i);
            }
        }

        vector<int> topoOrder;
        while (!q.empty()) {
            int u = q.front();
            q.pop();
            topoOrder.push_back(u);

            for (int v : adj[u]) {
                inDegree[v]--;
                if (inDegree[v] == 0) {
                    q.push(v);
                }
            }
        }

        if ((int)topoOrder.size() < V) {
            // Cycle detected: not all vertices could be sorted
            return {};
        }

        return topoOrder;
    }
};

int main() {
    // 1. DAG Test (Acyclic Course Schedule)
    // 5 tasks: 0 -> 2, 1 -> 2, 2 -> 3, 2 -> 4, 3 -> 4
    cout << "--- Kahn's Algorithm on DAG ---\n";
    KahnGraph g1(5);
    g1.addEdge(0, 2);
    g1.addEdge(1, 2);
    g1.addEdge(2, 3);
    g1.addEdge(2, 4);
    g1.addEdge(3, 4);

    auto order = g1.topologicalSort();
    if (order.empty()) {
        cout << "Cycle detected! No topological order exists.\n";
    } else {
        cout << "Topological Order: ";
        for (int x : order) cout << x << " ";
        cout << "\n"; // Expected: 0 1 2 3 4 or 1 0 2 3 4
    }

    // 2. Cyclic Graph Test (Invalid Schedule)
    // 0 -> 1 -> 2 -> 0
    cout << "\n--- Kahn's Algorithm on Cyclic Graph ---\n";
    KahnGraph g2(3);
    g2.addEdge(0, 1);
    g2.addEdge(1, 2);
    g2.addEdge(2, 0);

    auto order2 = g2.topologicalSort();
    if (order2.empty()) {
        cout << "Cycle detected! No topological order exists.\n";
    } else {
        cout << "Topological Order: ";
        for (int x : order2) cout << x << " ";
        cout << "\n";
    }

    return 0;
}

In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 2. Topological Sort — DFS-Based Approach

### The Why
A DFS-based topological sort is built on the concept of **post-order traversal**. By recursing all the way to the end of a path and then printing nodes on backtracking, we ensure that a node's dependencies are all processed *before* the node itself is finalized.

### Core Mechanism
1. **DFS Traversal:** For each unvisited node, run DFS.
2. **Backtracking Stack:** In the DFS function, traverse all neighbors of the current node $u$. After finishing all recursive calls for $u$'s neighbors, push $u$ to a stack or vector.
3. **Reversal:** The stack will store nodes in reverse topological order (deepest sinks are pushed first). Reversing this list gives the correct topological order.
4. **Cycle Detection:** Just like standard DFS on directed graphs, we *must* use **3-state coloring** (White/0, Gray/1, Black/2) to detect back-edges. If we encounter a node in the Gray state during traversal, a cycle exists, and topological sort is impossible.

### Common Pitfalls
- **Skipping Cycle Detection:** A naive DFS post-order traversal will happily output a sequence for cyclic graphs, but the sequence will be completely invalid. You *must* implement cycle detection (3-state coloring) alongside the DFS traversal.

In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <algorithm>

using namespace std;

class DFSTopoGraph {
    int V;
    vector<vector<int>> adj;

    // 3-state coloring: 0 = White, 1 = Gray, 2 = Black
    bool dfs(int u, vector<int>& state, vector<int>& order) {
        state[u] = 1; // Mark as Gray (visiting)

        for (int v : adj[u]) {
            if (state[v] == 1) {
                return true; // Cycle detected!
            }
            if (state[v] == 0) {
                if (dfs(v, state, order)) {
                    return true;
                }
            }
        }

        state[u] = 2; // Mark as Black (fully processed)
        order.push_back(u); // Post-order insertion
        return false;
    }

public:
    DFSTopoGraph(int vertices) : V(vertices) {
        adj.resize(V);
    }

    void addEdge(int u, int v) {
        adj[u].push_back(v);
    }

    // Returns empty vector if cycle detected
    vector<int> topologicalSort() {
        vector<int> state(V, 0);
        vector<int> order;

        for (int i = 0; i < V; ++i) {
            if (state[i] == 0) {
                if (dfs(i, state, order)) {
                    return {}; // Cycle detected
                }
            }
        }

        // Reverse the post-order sequence to obtain topological order
        reverse(order.begin(), order.end());
        return order;
    }
};

int main() {
    cout << "--- DFS-Based Topological Sort on DAG ---\n";
    DFSTopoGraph g1(5);
    g1.addEdge(0, 2);
    g1.addEdge(1, 2);
    g1.addEdge(2, 3);
    g1.addEdge(2, 4);
    g1.addEdge(3, 4);

    auto order = g1.topologicalSort();
    if (order.empty()) {
        cout << "Cycle detected! No topological order exists.\n";
    } else {
        cout << "Topological Order: ";
        for (int x : order) cout << x << " ";
        cout << "\n";
    }

    cout << "\n--- DFS-Based Topological Sort on Cyclic Graph ---\n";
    DFSTopoGraph g2(3);
    g2.addEdge(0, 1);
    g2.addEdge(1, 2);
    g2.addEdge(2, 0);

    auto order2 = g2.topologicalSort();
    if (order2.empty()) {
        cout << "Cycle detected! No topological order exists.\n";
    } else {
        cout << "Topological Order: ";
        for (int x : order2) cout << x << " ";
        cout << "\n";
    }

    return 0;
}

In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 3. Disjoint Set Union (DSU) — Core Operations

### The Why
Consider a dynamic scenario: you have $N$ isolated objects, and you want to repeatedly execute two commands:
1. **Union:** Connect two objects.
2. **Find:** Query if there is a path connecting two objects.

If we represent this with standard graph traversals (BFS/DFS), each query would cost $O(V+E)$. If we perform $Q$ queries on $N$ elements, the total complexity is $O(Q \cdot (V+E))$, which is far too slow. 
**Disjoint Set Union (DSU)** solves this by organizing objects into trees where each set has a "representative" (the root of the tree). Using two clever optimizations, DSU achieves an amortized cost of $O(\alpha(N))$ per operation, where $\alpha(N)$ is the **Inverse Ackermann function** (which is less than 5 for any realistic input size).

### Core Mechanism

#### 1. Representation
We store a `parent` array where `parent[i]` points to the parent of node $i$. Initially, every element is its own parent: `parent[i] = i`.

#### 2. Find with Path Compression
To find which set an element belongs to, we recursively traverse up to the root.
**Path Compression Optimization:** During the traversal, we update `parent[i]` to point *directly* to the root. This flattens the tree dynamically, ensuring subsequent lookups are $O(1)$!

```
     Naive Lookup Path:              With Path Compression:
          (Root)                             (Root)
            |                               /  |  \
           (A)                            (A) (B) (C)
            |
           (B)
            |
           (C)
```

#### 3. Union by Size or Rank
When merging two sets, we must decide which root points to which.
**Union by Size Optimization:** Always attach the root of the smaller tree under the root of the larger tree. This keeps the maximum tree depth logarithmic, preventing degenerated chains.

### Common Pitfalls
- **Incorrect Parent Assignment:** Assigning `parent[i] = j` instead of `parent[rootI] = rootJ`. You must always merge the *roots* of the sets, not the elements themselves. Merging elements directly breaks the tree structure.

In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <numeric>

using namespace std;

class DisjointSet {
    vector<int> parent;
    vector<int> size; // Union by Size optimization

public:
    DisjointSet(int n) {
        parent.resize(n);
        iota(parent.begin(), parent.end(), 0); // parent[i] = i
        size.assign(n, 1);
    }

    // Find operation with Path Compression
    int find(int i) {
        if (parent[i] == i) {
            return i;
        }
        // Path compression
        return parent[i] = find(parent[i]);
    }

    // Union operation by Size
    bool unite(int i, int j) {
        int rootI = find(i);
        int rootJ = find(j);

        if (rootI != rootJ) {
            // Merge smaller tree into larger tree
            if (size[rootI] < size[rootJ]) {
                swap(rootI, rootJ);
            }
            parent[rootJ] = rootI;
            size[rootI] += size[rootJ];
            return true; // Merged successfully
        }
        return false; // Already in same set
    }

    bool isConnected(int i, int j) {
        return find(i) == find(j);
    }

    int getSetSize(int i) {
        return size[find(i)];
    }
};

int main() {
    cout << "--- Disjoint Set Union (DSU) Demo ---\n";
    DisjointSet dsu(6);

    cout << "0 and 1 connected? " << (dsu.isConnected(0, 1) ? "Yes" : "No") << "\n";

    dsu.unite(0, 1);
    dsu.unite(1, 2);
    dsu.unite(3, 4);

    cout << "\nAfter unions (0-1, 1-2, 3-4):\n";
    cout << "0 and 2 connected? " << (dsu.isConnected(0, 2) ? "Yes" : "No") << "\n"; // Expected: Yes
    cout << "2 and 3 connected? " << (dsu.isConnected(2, 3) ? "Yes" : "No") << "\n"; // Expected: No
    cout << "Size of component containing 0: " << dsu.getSetSize(0) << "\n";       // Expected: 3

    dsu.unite(2, 3);
    cout << "\nAfter union (2-3):\n";
    cout << "2 and 3 connected? " << (dsu.isConnected(2, 3) ? "Yes" : "No") << "\n"; // Expected: Yes
    cout << "0 and 4 connected? " << (dsu.isConnected(0, 4) ? "Yes" : "No") << "\n"; // Expected: Yes
    cout << "Size of component containing 4: " << dsu.getSetSize(4) << "\n";       // Expected: 5

    return 0;
}

In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 4. DSU Applications — Undirected Cycle Detection

### The Why
In undirected graphs, running a DFS to find cycles requires traversal over the whole graph, which is an $O(V+E)$ operations step. DSU allows us to check for cycles incrementally on the fly as edges are added, which is highly useful in stream processing and algorithms like Kruskal's.

### Core Mechanism
We process each undirected edge `(u, v)` one-by-one:
1. Query `rootU = find(u)` and `rootV = find(v)`.
2. If `rootU == rootV`, they are already connected through some path. Adding the edge `(u, v)` would close a loop, meaning a **cycle is detected**.
3. If `rootU != rootV`, no cycle is closed. We safely merge the sets: `unite(u, v)`.

```
  Step 1: Edge (0, 1) -> Merge 0 and 1
  Step 2: Edge (1, 2) -> Merge 1 and 2 (connected: 0-1-2)
  Step 3: Edge (2, 0) -> find(2) == find(0) -> Cycle closed!
```

### Common Pitfalls
- **Directed cycle error:** DSU cycle detection does *not* work for directed graphs. If you have edges $0 \rightarrow 1$ and $0 \rightarrow 2$ and $1 \rightarrow 2$, DSU will report a cycle (since 1 and 2 are in the same set), but no directed cycle actually exists.

In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <numeric>

using namespace std;

class CycleDSU {
    vector<int> parent;
    vector<int> rank;

public:
    CycleDSU(int n) {
        parent.resize(n);
        iota(parent.begin(), parent.end(), 0);
        rank.assign(n, 0);
    }

    int find(int i) {
        if (parent[i] == i) return i;
        return parent[i] = find(parent[i]); // path compression
    }

    bool unite(int i, int j) {
        int rootI = find(i);
        int rootJ = find(j);
        if (rootI == rootJ) return false;

        // Union by Rank
        if (rank[rootI] < rank[rootJ]) {
            parent[rootI] = rootJ;
        } else if (rank[rootI] > rank[rootJ]) {
            parent[rootJ] = rootI;
        } else {
            parent[rootJ] = rootI;
            rank[rootI]++;
        }
        return true;
    }
};

struct Edge {
    int u, v;
};

bool hasCycle(int V, const vector<Edge>& edges) {
    CycleDSU dsu(V);
    for (const auto& edge : edges) {
        if (dsu.find(edge.u) == dsu.find(edge.v)) {
            return true; // u and v already connected -> Cycle detected
        }
        dsu.unite(edge.u, edge.v);
    }
    return false;
}

int main() {
    cout << "--- Undirected Cycle Detection via DSU ---\n";
    vector<Edge> treeEdges = {{0, 1}, {1, 2}, {2, 3}};
    cout << "Tree has cycle? " << (hasCycle(4, treeEdges) ? "Yes" : "No") << "\n"; // Expected: No

    vector<Edge> cyclicEdges = {{0, 1}, {1, 2}, {2, 3}, {3, 0}};
    cout << "Cyclic graph has cycle? " << (hasCycle(4, cyclicEdges) ? "Yes" : "No") << "\n"; // Expected: Yes

    return 0;
}

In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 5. Benchmarking DSU — Path Compression Impact

### The Why
Why do we bother writing Path Compression and Union by Size? To prove that these optimizations are structurally necessary, we benchmark a naive DSU implementation against an optimized one using a worst-case input: sequential chaining.

### Core Mechanism
We perform $N = 150,000$ operations:
1. Union consecutive elements: `unite(i, i + 1)`.
2. For Naive DSU, this constructs a single long path of length $N$. Each `find(0)` call requires traversing $O(N)$ elements.
3. For Optimized DSU, the tree stays extremely flat due to path compression.

Let's compile and run the benchmark.

In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <chrono>
#include <numeric>

using namespace std;

struct Timer {
    string name;
    chrono::high_resolution_clock::time_point start;
    Timer(const string& n) : name(n), start(chrono::high_resolution_clock::now()) {}
    ~Timer() {
        auto end = chrono::high_resolution_clock::now();
        auto diff = chrono::duration_cast<chrono::milliseconds>(end - start).count();
        cout << name << ": " << diff << " ms\n";
    }
};

class NaiveDSU {
    vector<int> parent;
public:
    NaiveDSU(int n) {
        parent.resize(n);
        iota(parent.begin(), parent.end(), 0);
    }

    int find(int i) {
        while (parent[i] != i) {
            i = parent[i];
        }
        return i;
    }

    void unite(int i, int j) {
        int rootI = find(i);
        int rootJ = find(j);
        if (rootI != rootJ) {
            parent[rootI] = rootJ;
        }
    }
};

class OptimizedDSU {
    vector<int> parent;
    vector<int> size;
public:
    OptimizedDSU(int n) {
        parent.resize(n);
        iota(parent.begin(), parent.end(), 0);
        size.assign(n, 1);
    }

    int find(int i) {
        if (parent[i] == i) return i;
        return parent[i] = find(parent[i]); // Path compression
    }

    void unite(int i, int j) {
        int rootI = find(i);
        int rootJ = find(j);
        if (rootI != rootJ) {
            if (size[rootI] < size[rootJ]) swap(rootI, rootJ);
            parent[rootJ] = rootI;
            size[rootI] += size[rootJ];
        }
    }
};

int main() {
    const int N = 150000;
    
    cout << "--- DSU Performance Benchmark (N = " << N << ") ---\n";

    {
        NaiveDSU dsu(N);
        Timer t("Naive DSU (Linear operations)");
        for (int i = 0; i < N - 1; ++i) {
            dsu.unite(i, i + 1);
        }
        long long sum = 0;
        for (int k = 0; k < 1000; ++k) {
            sum += dsu.find(0);
        }
        cout << "Naive final root: " << sum/1000 << "\n";
    }

    {
        OptimizedDSU dsu(N);
        Timer t("Optimized DSU (Path Compression + Union by Size)");
        for (int i = 0; i < N - 1; ++i) {
            dsu.unite(i, i + 1);
        }
        long long sum = 0;
        for (int k = 0; k < 1000000; ++k) { // 1,000,000 lookups to highlight performance
            sum += dsu.find(0);
        }
        cout << "Optimized final root: " << sum/1000000 << "\n";
    }

    return 0;
}

In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 6. Checkpoint, Cheat Sheet, and Answer Key

### Algorithmic Cheat Sheet

| Algorithm / Data Structure | Time Complexity | Space Complexity | Cycle Detection Capability |
|---|---|---|---|
| **Kahn's (BFS Topo Sort)** | $O(V + E)$ | $O(V + E)$ | Detects directed cycles if processed count < $V$. |
| **DFS-Based Topo Sort** | $O(V + E)$ | $O(V + E)$ | Detects directed cycles via 3-state coloring. |
| **Naive DSU** | $O(N)$ worst-case | $O(N)$ | Detects undirected cycles. |
| **Optimized DSU** | $O(\alpha(N))$ amortized | $O(N)$ | Detects undirected cycles on the fly. |

### Checkpoint Questions
1. Why does Kahn's algorithm require an explicit tracking of indegrees, whereas DFS-based topological sort does not?
2. What is the physical limit of the Inverse Ackermann function $\alpha(N)$ for any input size in the observable universe?
3. Why does Naive DSU degenerate to $O(N)$ time per operation, and how does Path Compression prevent this?
4. In Kahn's algorithm, what happens to the output if the input graph contains a directed cycle?
5. Why is DSU inappropriate for directed cycle detection?
6. Can topological sorting have multiple valid outputs for a single DAG?

### Answer Key
1. Kahn's algorithm processes nodes from the "front" (nodes with indegree = 0) and must know when a node has no remaining prerequisites. DFS-based topological sort works from the "back" (post-order traversal), pushing nodes to the stack only after resolving all their descendants.
2. $\alpha(N) \le 4$ for all practical inputs (since $A(4,4)$ is a number far larger than the number of atoms in the universe). Thus, for all practical purposes, DSU operations run in $O(1)$ amortized time.
3. Naive DSU links roots arbitrarily, which can form a single long linear tree (like a linked list) of depth $N$. Path compression flattens the tree by pointing visited nodes directly to the root, ensuring future operations take $O(1)$ steps.
4. Kahn's algorithm will output a truncated topological order containing only vertices that do not depend on the cycle. The size of the output list will be strictly less than $V$.
5. DSU does not track directed edges. If we have directed edges $A \rightarrow B$ and $B \rightarrow C$ and $A \rightarrow C$, DSU will report a cycle (since they are all in the same component), even though a DAG contains no directed cycles.
6. Yes. If multiple vertices have an indegree of 0 at the same time, they can be processed in any order, resulting in different valid topological sequences.

### Mini Project: Social Network Group Dynamics
Manage friendship connections in a social network dynamically. 
Using DSU:
1. Merge components when friendships are made (`unite`).
2. Track the number of isolated components and the size of the largest group in $O(\alpha(N))$ time.
3. Validate if two arbitrary users are connected via any path.

In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <string>
#include <unordered_map>
#include <numeric>
#include <algorithm>

using namespace std;

class SocialDSU {
    vector<int> parent;
    vector<int> size;
    int numComponents;
    int maxComponentSize;

public:
    SocialDSU(int n) {
        parent.resize(n);
        iota(parent.begin(), parent.end(), 0);
        size.assign(n, 1);
        numComponents = n;
        maxComponentSize = 1;
    }

    int find(int i) {
        if (parent[i] == i) return i;
        return parent[i] = find(parent[i]);
    }

    bool unite(int i, int j) {
        int rootI = find(i);
        int rootJ = find(j);
        if (rootI == rootJ) return false;

        if (size[rootI] < size[rootJ]) swap(rootI, rootJ);
        parent[rootJ] = rootI;
        size[rootI] += size[rootJ];

        numComponents--;
        maxComponentSize = max(maxComponentSize, size[rootI]);
        return true;
    }

    int getNumComponents() const { return numComponents; }
    int getMaxComponentSize() const { return maxComponentSize; }
    bool isConnected(int i, int j) { return find(i) == find(j); }
};

int main() {
    vector<string> people = {"Alice", "Bob", "Charlie", "David", "Eve", "Frank"};
    int n = people.size();
    unordered_map<string, int> nameToId;
    for (int i = 0; i < n; ++i) {
        nameToId[people[i]] = i;
    }

    SocialDSU social(n);

    cout << "--- Dynamic Friendships Simulation ---\n";
    cout << "Initial groups: " << social.getNumComponents() << ", Max group size: " << social.getMaxComponentSize() << "\n";

    social.unite(nameToId["Alice"], nameToId["Bob"]);
    social.unite(nameToId["Charlie"], nameToId["David"]);

    cout << "\nAfter friendships (Alice-Bob, Charlie-David):\n";
    cout << "Groups: " << social.getNumComponents() << ", Max group size: " << social.getMaxComponentSize() << "\n";
    cout << "Is Alice connected to Charlie? " << (social.isConnected(nameToId["Alice"], nameToId["Charlie"]) ? "Yes" : "No") << "\n";

    social.unite(nameToId["Bob"], nameToId["Charlie"]);
    cout << "\nAfter Bob-Charlie connection:\n";
    cout << "Groups: " << social.getNumComponents() << ", Max group size: " << social.getMaxComponentSize() << "\n";
    cout << "Is Alice connected to Charlie now? " << (social.isConnected(nameToId["Alice"], nameToId["Charlie"]) ? "Yes" : "No") << "\n";

    return 0;
}

In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 7. LeetCode Practice Problems

| # | Problem | Difficulty | Topic | Hint |
|---|---|---|---|---|
| 207 | Course Schedule | Medium | Topological Sort | Standard directed cycle detection. Can use Kahn's or DFS coloring. |
| 210 | Course Schedule II | Medium | Topological Sort | Same as 207, but return the completed topological sequence. If a cycle exists, return an empty array. |
| 310 | Minimum Height Trees | Medium | Kahn's Variant | Run a BFS peeling process similar to topological sorting, removing leaf nodes (degree = 1) step-by-step until only the centroid remains. |
| 684 | Redundant Connection | Medium | DSU | Find the first edge in an undirected graph that connects two vertices already in the same DSU component. |
| 547 | Number of Provinces | Medium | DSU / DFS | Can be solved with DSU: run union on all edges and return the final number of components. |
| 990 | Satisfiability of Equality Equations | Medium | DSU | First, merge variables connected by `==` via DSU. Then, check if any `!=` constraint links variables in the same component. |
| 721 | Accounts Merge | Medium | DSU | Represent email ownership as sets. Union email addresses that share accounts, then group them by their DSU representatives. |
| 323 | Number of Connected Components in an Undirected Graph | Medium | DSU / DFS | Perform DSU union for each edge; the final answer is the number of distinct roots. |
| 827 | Making A Large Island | Hard | DSU Grid | Assign island cells to component IDs using DSU, keeping track of component sizes. Then, try toggling each water cell (`0` to `1`) and sum adjacent component sizes. |

### Self-Check Before Moving to `03_Shortest_Paths`
- Can you write Kahn's topological sort from memory?
- Do you understand the difference between Directed Cycle Detection (DFS 3-state) and Undirected Cycle Detection (DSU)?
- Can you explain why path compression is critical for DSU efficiency?

If you feel confident, progress to **03_Shortest_Paths.ipynb**, where we explore shortest path algorithms (Dijkstra, Bellman-Ford, and Floyd-Warshall) and their respective trade-offs on weighted graphs.